# Fine-tuning a whole agent crew — LoRA & QLoRA

**One 80 MB adapter that teaches an open 7B how seven different AI agents are
supposed to answer.**

[QDev Orchestration AI](https://rohitkumarmanne-442.github.io/qdev-crew-adapter/)
runs an SDLC pipeline where each stage is a named agent: Phineas drafts the
story, Ferb proposes the code change, Brain reviews the diff, Velma authors the
test plan, Scooby files bugs, Perry plans the release. Every one of them has to
return a **strict JSON object** the platform can parse — right fields, right
enum values, enough substance to act on.

A base 7B given that contract mostly writes prose instead. This notebook fixes
that, and measures the fix honestly.

## What actually happened

| | before | after |
|---|---|---|
| **Crew score** | 0.528 | **0.907** |
| On real production work | 0.617 | **0.954** |
| Responses that parsed | 44 / 74 | **71 / 74** |
| Brain (code review) | 0.000 | **1.000** |
| Scooby (bug filing) | 0.250 | **1.000** |
| Velma (test authoring) | 0.272 | **0.844** |

Scored on 74 held-out prompts. **18 of them are real production artifacts** —
stories a human actually wrote and the platform actually answered — replayed
through the exact prompt builders that run in production. The adapter never sees
them during training.

Cost: $28.86 of teacher generation, 17 hours on one rented L4.

## How the measurement works

The scorer (`evaluate/scorer.py`, shipped in the bundle) is **deterministic and
offline** — no judge model, no sampling. A response earns credit for four
things: schema conformance (0.30), valid enum values (0.20), content rigor
(0.35), and absence of filler like `TODO` (0.15).

It rates the platform's own production output **0.999**, which is the check that
makes it a ceiling rather than a curve. If it couldn't score known-good work
near 1.0, the scorer would be the broken thing.

Every stage below uses the **same eval set, the same scorer and greedy
decoding**. That identity is the only reason a before/after means anything.

## Before you start

**Runtime → Change runtime type → L4 GPU.** Buying Colab Pro does *not* change
the runtime type by itself, and Pro does *not* include background execution —
that's Pro+. Cell 1 checks, and the script refuses to start on a T4 rather than
quietly producing an adapter that can't do QE.

Since the tab must stay open, **both long stages are resumable** (cell 2b).
After a disconnect: re-run cells 1, 2, 2b, then the cell that died.

## Timing — measured on an L4, not estimated

| cell | stage | time |
|---|---|---|
| 1–2b | GPU check, install, upload, Drive | ~6 min |
| 3 | smoke test (incl. 15 GB model download) | ~15 min |
| 4 | **baseline eval** | ~2.0 h |
| 5 | **QLoRA training** — 3 epochs, 7.9M tokens | **~5.7 h** |
| 6 | QLoRA eval | ~2.4 h |
| 7 | LoRA training (comparison arm) | ~4.4 h |
| 8 | LoRA eval | ~2.6 h |
| 9 | download | ~1 min |

Cells 1–6 are the headline, about 10 hours. Cells 7–8 are the LoRA-vs-QLoRA
comparison and can be a separate session.

Note that **evaluation cost more than training**: 7 hours across three arms
versus 5.7 hours to train. Measuring properly is the expensive part, and it's
the part usually skipped.

**Run every cell with no extra flags** unless a comment says otherwise. The
defaults are the full-quality ones.

## Why zephyr and not Qwen

On a 22 GB card the binding constraint isn't the 7B of weights — it's the
**logits tensor**, `vocab × seq`, held about three times over during a backward
pass:

| base | vocab | logits @12288 | outcome |
|---|---|---|---|
| Qwen2.5-7B | 152,064 | 7.0 GB | OOM |
| **zephyr-7b-beta** | **32,000** | **1.5 GB** | fits, 100% of the corpus |

Same 7B class, ungated, 4.75× smaller vocabulary. Zephyr is also Mistral-7B
architecture, so the run reports **41,943,040 trainable parameters** — matching
the arithmetic derived by hand from the model's dimensions.

The sequence window isn't a speed knob. At 4096, only 1 of 120 `qe` training
examples still fits its target, so a short window doesn't shrink the corpus
evenly — it deletes Velma's task entirely.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv
# ── CHECK THIS BEFORE GOING FURTHER ──────────────────────────────────────
#   WANT:  NVIDIA L4, 23034 MiB, 8.9     (or A100, 40960 MiB, 8.0)
#   WRONG: Tesla T4, 15360 MiB, 7.5
#
# Buying Colab Pro does NOT change your runtime type. If this says Tesla T4,
# go to Runtime -> Change runtime type -> L4 GPU -> Save. That restarts the
# runtime, so re-run this cell and the two below it afterwards.
#
# It is not a speed preference. Turing (7.5) has no flash-attention kernel, so
# attention memory is heads x seq^2 rather than linear, which caps the window
# at 4096 — and at 4096 only 1 of 120 qe training examples still fits its
# target. The adapter would never learn to author a test plan.

In [ ]:
%pip -q install -U "transformers>=4.57" "peft>=0.17" "trl>=0.24" \
  "bitsandbytes>=0.48" "accelerate>=1.10" "datasets>=3.0"
# liger-kernel is optional now — its fused cross-entropy matters for large-vocab
# models, and zephyr's 32k vocabulary makes the logits tensor small enough that
# it isn't needed. Uncomment to try it; the script probes it and falls back.
# %pip -q install -U "liger-kernel>=0.5"
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

## 2 · Upload the bundle

Locally, run `python -m colab.package` to produce `qdev_finetune_bundle.zip`
(3.2 MB), then upload it here. It carries `run_finetune.py`, the `evaluate/`
scorer and the four dataset splits.

The scorer travels *with* the data on purpose — the before/after comparison only
means something if both runs are graded by the identical function, so it ships
alongside rather than being re-implemented in a cell.

In [ ]:
from google.colab import files
import zipfile, pathlib, os, json
up = files.upload()
name = next(iter(up))
with zipfile.ZipFile(name) as z: z.extractall('.')
man = json.load(open('MANIFEST.json'))
for s, v in man['splits'].items():
    print(f"{s:<10}{v['n']:>5}   " + ' '.join(f'{k}={n}' for k, n in sorted(v['per_task'].items())))
# The hashes are here so that if a Colab number ever disagrees with a local one,
# "were the same bytes graded?" is answerable instead of debatable.
print('\nscorer sha256:', man['files']['evaluate/scorer.py']['sha256'])

## 2b · Persist `out/` to Drive — do not skip this

Colab **Pro does not include background execution** (that is Pro+). The run
depends on a browser tab staying open for up to 8 hours, so treat a disconnect
as likely rather than exceptional.

This points `out/` at Google Drive, which makes both long cells resumable:

- **evals** checkpoint after every batch. Re-run the same cell and it replays
  the finished rows from disk instead of regenerating them. Greedy decoding
  makes that exact — a resumed eval and an uninterrupted one produce identical
  numbers.
- **training** saves every 10 steps (~30 min). Re-run the cell and it continues
  from the newest checkpoint rather than step 0.

So after any disconnect: re-run cells 1, 2, 2b, then **the same cell that died**.

### Keeping the tab alive

- Stop the machine sleeping — Windows: Settings → System → Power → Screen and
  sleep → *Never* (on the "plugged in" setting)
- Leave the Colab tab open; a background tab is fine, a closed one is not
- Do not let the laptop lid close

The base model is **not** cached to Drive: zephyr is ~15 GB and free Drive is 15
GB total. A fresh runtime re-downloads it in ~40 s on an L4.

In [ ]:
import os, json, pathlib
from google.colab import drive
drive.mount('/content/drive')

DEST = pathlib.Path('/content/drive/MyDrive/qdev_finetune/out')
DEST.mkdir(parents=True, exist_ok=True)

# run_finetune.py writes to the relative path "out". Symlinking it means the
# script needs no Drive awareness at all — and `mkdir(exist_ok=True)` on a
# symlink pointing at a live directory is a no-op, so nothing breaks.
if pathlib.Path('out').is_symlink():
    os.unlink('out')
elif pathlib.Path('out').exists():
    raise SystemExit("a real ./out already exists — move it before symlinking, "
                     "or you will lose whatever is in it")
os.symlink(DEST, 'out')

res = DEST / 'results.json'
done = sorted(json.load(res.open()).keys()) if res.exists() else []
print(f"out/ -> {DEST}")
print("stages already recorded:", ', '.join(done) if done else "none (fresh start)")
print("adapters on Drive:", [p.name for p in DEST.iterdir() if p.is_dir()] or "none")

## 3 · Smoke test — run this before anything expensive

Four training steps on the **eight longest examples in the corpus** (peak memory
is set by the longest sequence, so a smoke test on average rows proves nothing),
then one eval row per task at a 1200-token cap.

Includes a one-time ~15 GB model download.

### Reading the result

```
GPU NVIDIA L4  sm_89  22.0 GiB
seq 12288: needs 7.1 GiB (attn 0.2 + acts 3.0 + logits 2.9 + grads/opt 1.0)  fits
seq window 12288
trainable 41,943,040 / ... = 1.11%
```

`attn 0.2` is flash attention — on a T4 that same line reads `attn 18.0`.
`12288` means no training example was dropped. `41,943,040` matches
`calc/report.py`.

Then the per-task eval table. **`bug`, `review` and `plan` should show
`emit` above 0** — their objects fit inside 1200 tokens, so a non-zero emit
proves the chat template, generation and JSON extraction all work. `qe` and
`dev` will truncate and read 0.00; that is the cap, not a fault.

If **every** task shows `emit 0.00`, the script says so explicitly and points at
`out/raw_*.json`. Truncated JSON there is fine. Prose or empty strings are not —
stop and investigate rather than spending three hours measuring a broken harness.

In [ ]:
# Saves to out/smoke, NOT out/qlora — so a half-trained 4-step adapter can never
# be mistaken for the real one if a later cell fails.
!python run_finetune.py --stage qlora --smoke --name smoke && \
 python run_finetune.py --stage eval --adapter out/smoke --name wiring --smoke --batch 2 && \
 echo "SMOKE OK — the long cells will run"

## 4 · Baseline — scoring the untouched model

This is the ***before*** number, and it's the one that has to be honest. Nothing
has been trained yet. Same 74 rows, same scorer, same greedy decoding, same
per-task generation ceilings as every later stage.

**What the eval set is:** 18 real production artifacts (stories a human wrote
and the platform answered, replayed through the exact production prompt
builders) plus 56 generated rows covering the five tasks with no production data
yet. The adapter never sees any of them in training.

**One thing that had to be fixed first.** An early version of this prompt ended
with *"call the `file_bug` tool"* — correct in production, where the model runs
with tools bound. But an open model here has no tools, so it was being told to
invoke something that doesn't exist, and it wrote prose. Every task scored 0.000.

That would have been a *measurement of a prompt bug*, not of the model, and a
0.00 → 0.95 chart resting on it would be indefensible. The prompt now states the
JSON contract in plain language, rendered from the schema so instruction and
grader can't drift apart.

With a fair prompt the baseline lands at **0.528**, not zero. A 7B given an
explicit contract complies about half the time — and *that* is a number nobody
can accuse you of rigging.

Expect this to be the slowest cell (~2 h). A base model rarely emits EOS, so it
runs to the generation ceiling on most rows. The tuned model stops when its
object is closed, which is itself evidence the training took.

In [ ]:
# No flags: the defaults are now the full-quality ones (74 eval rows, batch 4,
# window auto-sized from measured VRAM). Passing --per-task 3 here would silently
# re-impose the T4 compromise.
!python run_finetune.py --stage baseline

## 5 · QLoRA training

Three techniques stacked, and each one earns its place:

**4-bit NF4** for the frozen base. NF4 places its 16 levels at the *quantiles*
of a normal distribution, so each level carries 1/16 = 6.25% of the probability
mass. Weights are roughly normal, so resolution lands near zero where they
actually are — measured 14.6% lower error than evenly-spaced int4 on the same
weights.

**Double quantization** — the per-block scales get quantized too, taking scale
overhead from `32/64 = 0.500` bits per weight down to `8/64 + 32/(64×256) =
0.127`. A "4-bit" weight really costs 4.127 bits.

**Paged 8-bit AdamW**, so optimiser state can spill to host memory instead of
OOMing at a spike.

Three epochs over the full **827-example corpus**, all seven tasks, nothing
dropped.

### What to watch

The `trainable / total` line is the LoRA argument in one number:

```
trainable 41,943,040 / 3,794,014,208 = 1.1055%
```

Don't be confused by that percentage — bitsandbytes packs two 4-bit weights per
`uint8`, so `numel()` reports roughly half the real parameter count. The honest
figure is **41,943,040 / 7,248,000,000 = 0.58%**, and cell 7's bf16 arm prints
exactly that.

Also expect validation loss to *rise* slightly across epochs (0.828 → 0.873)
while token accuracy stays flat and entropy drops sharply. That's the model
getting **more confident**, not more wrong — and for structured output that
sharpening is what makes it reliably close its JSON. The scorer is the arbiter,
not eval loss.

In [ ]:
!python run_finetune.py --stage qlora --rank 16 --alpha 32
# rank 16 / alpha 32 is deliberate, not a default left alone: it is the exact
# configuration calc/report.py models, so the adapter this produces is the one
# the LinkedIn arithmetic describes. epochs defaults to 3.

In [ ]:
!python run_finetune.py --stage eval --adapter out/qlora --name qlora
# 6 · The AFTER number. Identical eval set, scorer, ceilings and greedy decoding
# as the baseline — that identity is the only reason the comparison means
# anything.

## 7 · LoRA — the comparison arm

Everything identical to cell 5 — same rank, same corpus, same optimiser, same
schedule — except the frozen base stays in **bf16 instead of 4-bit NF4**. One
variable, so any difference is attributable.

### The window has to be 8192 here, and that IS the result

A bf16 base is 13.7 GB resident against QLoRA's 4.9 GB. At the 12,288-token
window QLoRA trains at, this arm **runs out of memory on a 22 GB L4** — it dies
around step 6 of 156, roughly ten minutes in. 8192 is the largest window it can
hold.

That isn't a footnote to apologise for, it's the finding:

> Quantizing the frozen base to 4 bits cost **no measurable quality**. What it
> bought was **room** — 50% more context on the same card.

Measured outcome across the two arms:

| | QLoRA | LoRA |
|---|---|---|
| resident VRAM | **4.9 GiB** | 13.7 GiB |
| window it could afford | **12,288** | 8,192 |
| corpus it trained on | 100% | 98% (`dev` 87%) |
| overall score | **0.907** | 0.846 |

On the three tasks where both arms saw identical data — `bug`, `devops`,
`review` — they tie at exactly 1.000. The whole gap sits in `qe` and `dev`, the
long-target tasks where the shorter window cost LoRA training examples.

So this is **not** a controlled quantization A/B. The windows differ, and that
difference is precisely what the comparison is measuring.

In [ ]:
# --max-length 8192 is REQUIRED here, not optional. A bf16 base leaves ~8.3 GB
# free on a 22 GB L4, and the 12,288 window QLoRA uses needs more than that —
# this arm OOMs at step 6 of 156 without it. Pinning the window explicitly means
# you either get a valid run or a loud failure, never a silent step-down to a
# smaller corpus that would quietly break the comparison.
!python run_finetune.py --stage lora --rank 16 --alpha 32 --max-length 8192

In [ ]:
!python run_finetune.py --stage eval --adapter out/lora --name lora

## 9 · Download the results

`results.json` holds every score. `raw_*.json` holds the actual model responses,
which is what makes a before/after side-by-side possible — you can put the base
model's prose next to the adapter's object for the same prompt.

### Read the table with `dev` in mind

`dev` will score low in every arm, and it is **not** a model failure. Its
training targets emit the `changes` array first, carrying whole file bodies
(8,294 tokens at the 90th percentile), with `commit_message`, `pr_title`,
`pr_body` and `summary` all coming *after* it. Run out of generation budget
inside `changes` and four required fields vanish at once, which the scorer reads
as a schema failure rather than a truncated response.

The eval reports it explicitly:

```
! ran to the generation ceiling (output was cut off): dev 6/8, qe 1/16
```

The fix is a key reorder in the training targets so the cheap required scalars
come before the expensive array — then a retrain. Not applied here, because
changing the data would have invalidated the LoRA comparison in cells 7–8.

Worth stating plainly rather than hiding: a result with one diagnosed regression
is more believable than seven perfect scores.

In [ ]:
import json, shutil
res = json.load(open('out/results.json'))
# smoke_* rows are wiring checks at a 400-token ceiling — never report them.
real = {k: v for k, v in res.items() if not k.startswith('smoke_')}
print(f"{'run':<12}{'score':>8}{'emit':>8}{'schema':>8}{'enums':>8}{'rigor':>8}{'prod':>8}")
for run, v in real.items():
    o = v['overall']; r = v.get('by_slice', {}).get('real', {})
    print(f"{run:<12}{o['score']:>8.3f}{o['emit_rate']:>8.3f}{o['schema']:>8.3f}"
          f"{o['enums']:>8.3f}{o['rigor']:>8.3f}{r.get('score', float('nan')):>8.3f}")
if 'baseline' in real and 'qlora' in real:
    b, q = real['baseline']['overall']['score'], real['qlora']['overall']['score']
    print(f"\nbefore {b:.3f} -> after {q:.3f}   "
          f"+{q-b:.3f} absolute, {q/max(b, 1e-9):.1f}x")
shutil.make_archive('qdev_results', 'zip', 'out')
from google.colab import files; files.download('qdev_results.zip')